# Fraud detection: ETL + training pipeline

This notebook: **read from `shop.db` → clean and engineer features in memory → sklearn `Pipeline` → saved artifacts** for your web app.

- **Extract / transform:** load normalized tables from `shop.db`, denormalize to one row per order, engineer features into a single DataFrame `df` (kept **in memory**—no analytical SQLite file).
- **Target:** `is_fraud` on `orders` (0 = legitimate, 1 = fraud).
- **Features:** `risk_score` (0–100) plus order/customer aggregates. `is_fraud` is **only** the label.
- **Train:** split `df`, then impute → scale → logistic regression (`class_weight='balanced'`), save `fraud_model.sav`, `model_metadata.json`, and `metrics.json`.

> In production you could still persist `df` or a warehouse table on a schedule; here training runs directly on the cleaned in-memory table.

## ETL — Extract and transform (in memory)

Build one modeling-ready DataFrame `df`; training uses it without writing to `warehouse.db`.

In [ ]:
import sqlite3
import pandas as pd

SHOP_DB = "shop.db"

conn = sqlite3.connect(SHOP_DB)

orders = pd.read_sql("SELECT * FROM orders", conn)
customers = pd.read_sql("SELECT * FROM customers", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)
products = pd.read_sql("SELECT * FROM products", conn)

conn.close()

print(orders.shape, customers.shape, order_items.shape, products.shape)

### Denormalize to one row per order

Aggregate line items, then join orders and customers. Order-level `risk_score` and `is_fraud` come from `orders`.

In [ ]:
order_item_features = (
    order_items.merge(products, on="product_id", how="left")
    .groupby("order_id")
    .agg(
        num_items=("quantity", "sum"),
        avg_unit_price=("unit_price", "mean"),
        total_value=("line_total", "sum"),
        avg_cost=("cost", "mean"),
    )
    .reset_index()
)

order_item_features.head()

In [ ]:
df = orders.merge(customers, on="customer_id", how="left").merge(
    order_item_features, on="order_id", how="left"
)

df.head()

### Features and label

In [ ]:
df["order_date"] = pd.to_datetime(df["order_datetime"])
df["birthdate"] = pd.to_datetime(df["birthdate"])
df["customer_age"] = (df["order_date"] - df["birthdate"]).dt.days // 365

df["customer_order_count"] = df.groupby("customer_id")["order_id"].transform("count")

df[["risk_score", "num_items", "total_value", "avg_cost", "customer_age", "customer_order_count"]].describe()

In [ ]:
df["is_fraud"] = df["is_fraud"].astype(int)

df["is_fraud"].value_counts(normalize=True)

## Training — use in-memory `df`, save artifacts

The cleaned `df` from above is split and fed to the pipeline (no reload from SQLite).

In [ ]:
print(df.shape)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

label_col = "is_fraud"

feature_cols = [
    "risk_score",
    "num_items",
    "total_value",
    "avg_cost",
    "customer_age",
    "customer_order_count",
]

X = df[feature_cols]
y = df[label_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(max_iter=1000, class_weight="balanced"),
        ),
    ]
)

pipeline

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

report = classification_report(y_test, y_pred, output_dict=True)

accuracy, f1, roc_auc

In [ ]:
import joblib

MODEL_PATH = "fraud_model.sav"
joblib.dump(pipeline, MODEL_PATH)
print(f"Saved: {MODEL_PATH}")

In [ ]:
import json
from datetime import datetime, timezone

model_version = "1.0.0"

metadata = {
    "model_name": "fraud_detection_pipeline",
    "model_version": model_version,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_data": "in_memory_after_etl",
    "num_training_rows": int(X_train.shape[0]),
    "num_test_rows": int(X_test.shape[0]),
    "features": feature_cols,
    "label": label_col,
}

metrics = {
    "accuracy": float(accuracy),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "classification_report": report,
}

with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Wrote model_metadata.json and metrics.json")

### Web app integration (smoke test)

Load `fraud_model.sav` and pass a DataFrame with **`feature_cols` in the same order**.

In [ ]:
loaded = joblib.load(MODEL_PATH)
sample = X_test.iloc[:3]
proba = loaded.predict_proba(sample)[:, 1]
list(zip(sample.index.tolist(), proba.round(4).tolist()))